In [ ]:
def get_the_right_cable(switch,Cables,absolute_sw_to_dev):
    cables = Cables[switch]
    higher_lengths = sorted([item for item in cables if isinstance(item.get('length'), 
                                    (int, float)) and item.get('length') > absolute_sw_to_dev], key=lambda x: x['length'])

    if higher_lengths:
        return higher_lengths[0]
    else:
        return {}

In [ ]:
def calculate_cable_lengths(rack, LANs, Cables, colors_info):
    unit_to_cm = 4.45
    # the following are in units
    device_to_rackside = 6.75
    racktop_to_ceiling = 4.5 # assumed 20cm
    rack_to_rack = 4.5 # assumed from side to the adjacent side no spacing
    lans = dict()
    devices = dict()
    for dev in rack:
        dev_id = dev.split('_')[-1]
        device = '_'.join(dev.split('_')[:-1])
        if 'LAN' in device:
            lanname, switch = dev.split('__')
            switch = '_'.join(switch.split('_')[:-1])
            lans[dev] = dict()
            device_specs = [x for x in LANs[lanname]['switch'] if x['model'] == switch ][0]
            lans[dev] = device_specs.copy()
            lans[dev]['position'] = rack[dev]
            lans[dev]['id'] = dev_id
            lans[dev]['half_for_split'] = 0
            lans[dev]['full_no_split'] = 0
            lans[dev]['cables'] = dict()
            lans[dev]['minimum_cable_total_length'] = 0
            lans[dev]['actual_cable_total_length'] = 0
            
    for lan in lans:
        lanname = lan.split('__')[0]
        for dev in rack:
            dev_id = dev.split('_')[-1]
            device = '_'.join(dev.split('_')[:-1])
            if 'LAN' not in device:
                if colors_info[device][lanname]['count'] > 0:
                    if dev not in devices:
                        devices[dev] = dict()
                    devices[dev]['position'] = rack[dev]
                    devices[dev]['id'] = dev_id
                    no_ports = colors_info[device][lanname]['count']
                    speed_ratio = lans[lan]['speed'] / colors_info[device][lanname]['speed']
                    device_position = rack[dev]
                    # absolute cable lengeth in meters:
                    absolute_sw_to_dev = ((lans[lan]['position'] - device_position) + 2*device_to_rackside)*unit_to_cm/100
                    cable = get_the_right_cable(switch,Cables,absolute_sw_to_dev)
                    no_of_cables = colors_info[device][lanname]['count']
                    devices[dev][lan] = dict()
                    devices[dev][lan]['cable_count'] = colors_info[device][lanname]['count']
                    devices[dev][lan]['speed'] = colors_info[device][lanname]['speed']
                    if speed_ratio < 1:
                        lans[lan]['half_for_split'] +=1
                        devices[dev][lan]['half_for_split'] = 1
                    else:
                        lans[lan]['full_no_split'] += 1
                        devices[dev][lan]['full_no_split'] = 1
                    devices[dev][lan]['cables_count'] = no_of_cables
                    devices[dev][lan]['cable_type'] = cable
                    if cable['model'] not in lans[lan]['cables']:
                        lans[lan]['cables'][cable['model']] = 0
                    lans[lan]['cables'][cable['model']] += no_of_cables
                    lans[lan]['minimum_cable_total_length'] += absolute_sw_to_dev * no_of_cables
                    lans[lan]['actual_cable_total_length'] += cable['length'] * no_of_cables
                   
                    
                    
                    
        
    
    return (devices, lans)

rack = {'LAN_1__IB+400_1': 41, 'FrontEnd_nodes_2': 40, 'GPU_nodes_3': 32, 'compute_nodes_4': 30, 'compute_nodes_5': 28, 'compute_nodes_6': 26, 'compute_nodes_7': 24, 'compute_nodes_8': 22}
lans, devices  = calculate_cable_lengths(rack, LANs, Cables, colors_info)
print(devices)
print(lans)

In [214]:
def sort_device_before_placement(all_items):
    lan_devices = sorted([item for item in all_items if 'LAN' in item['type']], key=lambda x: x['type'])
    other_devices = [item for item in all_items if 'LAN' not in item['type']]

    def sort_other(item):
        sort_key = [1] * len(lan_devices) + [item.get('type', '')] # Initialize with a lower priority and type for tie-breaking

        for i, lan_item in enumerate(lan_devices):
                for key, value in item.items():
                    if isinstance(value, dict):
                        sort_key[i] = -item.get('count', 0) # Higher count gets a higher (negative) priority
                        break # Move to the next LAN device after finding a match
        return tuple(sort_key)

    sorted_other_devices = sorted(other_devices, key=sort_other)
    return lan_devices + sorted_other_devices

all_servers = [{'model': 'IB+400', 'ports': 64, 'speed': 400, 'height': 2, 'wattage': 2000, 'weight': 20, 'type': 'LAN_3__IB+400'}, {'count': 20, 'wattage': 11000, 'height': 8, 'weight': 15, 'LAN_1': {'count':0, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 8, 'speed': 400}, 'type': 'GPU_nodes'}, {'count': 20, 'wattage': 1600, 'height': 2, 'weight': 30, 'LAN_1': {'count': 1, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 0, 'speed': 400}, 'type': 'compute_nodes'}, {'count': 20, 'wattage': 1600, 'height': 2, 'weight': 30, 'LAN_1': {'count': 1, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 0, 'speed': 400}, 'type': 'compute_nodes'}, {'count': 20, 'wattage': 1600, 'height': 2, 'weight': 30, 'LAN_1': {'count': 1, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 0, 'speed': 400}, 'type': 'compute_nodes'}, {'count': 20, 'wattage': 1600, 'height': 2, 'weight': 30, 'LAN_1': {'count': 1, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 0, 'speed': 400}, 'type': 'compute_nodes'}]

sorted_servers = sort_device_before_placement(all_servers)
print(sorted_servers)

In [201]:
def old_sort_device_before_placement(item):
    if 'LAN' in item['type']:
        return (0, item['type'])  # Prioritize LAN, then sort alphabetically by type
    else:
        return (1, item['type'])  # Other types come later, maintain original order



In [220]:
import numpy as np
from collections import Counter

def is_rack_stable(rack_weight, rack_height_mm, rack_cg_height_mm, servers, rack_depth_mm):
    """
    Checks if the rack configuration is likely stable based on the combined
    vertical center of gravity.
    """
    total_weight = rack_weight + sum(s[1] for s in servers)
    if total_weight == 0:
        return True

    combined_vertical_cg = (rack_weight * rack_cg_height_mm +
                             sum(s[1] * s[0] for s in servers)) / total_weight

    stability_threshold_fraction = 0.7  # Adjust as needed
    return combined_vertical_cg <= rack_height_mm * stability_threshold_fraction

def find_stable_positions_greedy_complex(rack_height_u, rack_weight_kg, rack_width_mm, rack_depth_mm,
                                         servers_to_place, LANs, Cables, server_specs, prioritize_top=False):
    """
    A greedy approach to find stable server positions for various server types inside one rack
    """
    u_height_mm = 44.45
    rack_height_mm = rack_height_u * u_height_mm
    rack_cg_height_mm = rack_height_mm / 2
    all_servers = []
    for server_type, count in servers_to_place.items():
        specs = server_specs.get(server_type)
        if not specs:
            if 'LAN' in server_type:
                lan , sw_model = server_type.split('__')
                
                specs = [x for x in LANs[lan]['switch'] if x['model'] == sw_model][0]
                
            else:
                print(f"Warning: Specifications not found for server type '{server_type}'. Skipping.")
                continue
           
                
        specs['type'] = server_type
        for _ in range(count):
                all_servers.append(specs)

    if not all_servers:
        return {}  # Return empty dict for empty Counter
    all_servers = sort_device_before_placement(all_servers)
    best_placement = ()
    best_cable_lengths = float(inf)
    if not prioritize_top:
        placed_servers_info = []
        occupied_u = [False] * rack_height_u
        for server in all_servers:
            server_height_u = server['height']
            server_weight_kg = server['weight']
            server_cg_offset = (server_height_u * u_height_mm) / 2
            placed = False
            for i in range(rack_height_u):
                if not occupied_u[i]:
                    start_u = i
                    server_base_height = start_u * u_height_mm
                    server_cg = server_base_height + server_cg_offset
                    can_place = True
                    for u_check in range(start_u, start_u + server_height_u):
                        if u_check >= rack_height_u or occupied_u[u_check]:
                            can_place = False
                            break
                    if can_place:
                        temp_positions = [(p['cg'], p['weight']) for p in placed_servers_info] + [(server_cg, server_weight_kg)]
                        if is_rack_stable(rack_weight_kg, rack_height_mm, rack_cg_height_mm, temp_positions, rack_depth_mm):
                            placed_servers_info.append({'cg': server_cg, 'weight': server_weight_kg, 'type': server['type'], 'height': server['height'], 'start_u': start_u})
                            for u in range(start_u, start_u + server_height_u):
                                if u < rack_height_u:
                                    occupied_u[u] = True
                            placed = True
                            break
            if not placed:
                return None, None, None

        final_placement = {}
        for i, server_info in enumerate(placed_servers_info):
            final_placement[f"{server_info['type']}_{i+1}"] = server_info['start_u'] + 1
    else:
        placed_servers_info = []
        occupied_u = [False] * rack_height_u
        for server in all_servers:
            server_height_u = server['height']
            server_weight_kg = server['weight']
            server_cg_offset = (server_height_u * u_height_mm) / 2
            placed = False
            for i in range(rack_height_u - 1, -1, -1):
                if not occupied_u[i]:
                    start_u = i
                    server_base_height = start_u * u_height_mm
                    server_cg = server_base_height + server_cg_offset
                    can_place = True
                    for u_check in range(start_u, start_u + server_height_u):
                        if u_check >= rack_height_u or occupied_u[u_check]:
                            can_place = False
                            break
                    if can_place:
                        temp_positions = [(p['cg'], p['weight']) for p in placed_servers_info] + [(server_cg, server_weight_kg)]
                        if is_rack_stable(rack_weight_kg, rack_height_mm, rack_cg_height_mm, temp_positions, rack_depth_mm):
                            placed_servers_info.append({'cg': server_cg, 'weight': server_weight_kg, 'type': server['type'], 'height': server['height'], 'start_u': start_u})
                            for u in range(start_u, start_u + server_height_u):
                                if u < rack_height_u:
                                    occupied_u[u] = True
                            placed = True
                            break
            if not placed:
                return None, None, None

        final_placement = {}
        sorted_servers = sorted(placed_servers_info, key=lambda x: x['start_u'], reverse=True)
        occupied_map = [False] * rack_height_u
        placed_index = 1
        for server in sorted_servers:
            start_u = server['start_u']
            server_type = server['type']
            server_height = server['height']
            for u in range(start_u, -1, -1):
                can_place_here = True
                for check_u in range(u, u + server_height):
                    if check_u >= rack_height_u or occupied_map[check_u]:
                        can_place_here = False
                        break
                if can_place_here:
                    final_placement[f"{server_type}_{placed_index}"] = u + 1
                    for occupy_u in range(u, u + server_height):
                        if occupy_u < rack_height_u:
                            occupied_map[occupy_u] = True
                    placed_index += 1
                    break
            else:
                # Fallback to the initially found stable position
                final_placement[f"{server_type}_{placed_index}"] = start_u + 1
                for occupy_u in range(start_u, start_u + server_height):
                    if occupy_u < rack_height_u:
                        occupied_map[occupy_u] = True
                placed_index += 1
        #print('-->',final_placement)
    
    devices, lans = calculate_cable_lengths(final_placement,LANs, Cables,colors_info)
    if get_cable_lengths < best_cable_lengths:
        best_placement = (final_placement, devices,lans)
        best_cable_lengths = get_cable_lengths
    return best_placement
    


# --- Example Usage ---
rack_height_u = 42
rack_weight_kg = 114.55
rack_width_mm = 600
rack_depth_mm = 1200

all_racks_config = [
    Counter({'LAN_1__IB+400':1,'compute_nodes': 5, 'GPU_nodes': 1, 'FrontEnd_nodes': 1}),
    Counter({'FrontEnd_nodes': 12, 'GPU_nodes': 1}),
    Counter({'compute_nodes': 5, 'GPU_nodes': 1, 'FrontEnd_nodes': 1}),
    Counter({'FrontEnd_nodes': 15, 'compute_nodes': 5}),
    Counter({'compute_nodes': 5, 'GPU_nodes': 1, 'FrontEnd_nodes': 1}),
    Counter({'storage_nodes': 8}),
    Counter({'storage_nodes': 15}),
    Counter({'storage_nodes': 7, 'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter(),
    Counter(),
]


colors_info = {
        'compute_nodes': {'count': 20, 'wattage': 1600, 'height': 2, 'weight':30, 'LAN_1':{'count':1,'speed':400},
                                                                           'LAN_2':{'count':1,'speed':200},
                                                                           'LAN_3':{'count':1,'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                            'LAN_6':{'count':0, 'speed':400},
                      },
        'GPU_nodes': {'count': 20, 'wattage': 11000, 'height': 8, 'weight': 15, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':1,'speed':200},
                                                                           'LAN_3':{'count':1, 'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                            'LAN_6':{'count':8, 'speed':400},
                      },
        'storage_nodes': {'count': 30, 'wattage': 1250, 'height': 2, 'weight':20, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':3,'speed':200},
                                                                           'LAN_3':{'count':1, 'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                          'LAN_6':{'count':0, 'speed':400},
                         },
        'FrontEnd_nodes': {'count': 30, 'wattage': 750, 'height': 1, 'weight':9, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':0,'speed':200},
                                                                            'LAN_3':{'count':1, 'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                         'LAN_6':{'count':0, 'speed':400},
                          },
                   
    }
LANs = { 'LAN_1':{'type':'400gbsNDR','topology': 'halfports_spine_leaf',
                  'switch':[{'model':'IB+400','ports':64,'speed':400, 'height':2,'wattage':2000, 'weight':20},
                         {'model':'IB_800','ports':64,'speed':800, 'height':2,'wattage':2000, 'weight':20},
                           ]},
              'LAN_2':{'type':'400GbpsEth', 'topology': 'uplinks_spine_leaf',
'switch':[{'model':'Z_9xxx','ports':64,'speed':400, 'height':2,'wattage':2200, 'uplink_count':4, 'uplink_speed':800,'weight':20},
                 {'model':'Z_6xxx','ports':64,'speed':200, 'height':1,'wattage':2000, 'uplink_count':4, 'uplink_speed':400,'weight':20},
        ]},
              'LAN_3':{'type':'25gbps','topology': 'uplinks_spine_leaf',
        'switch':[{'model':'S_xx64','ports':64,'speed':25, 'height':2,'wattage':300, 'uplink_count':4, 'uplink_speed':100,'weight':2},
                {'model':'S_xx32','ports':32,'speed':25, 'height':1,'wattage':200,'uplink_count':4, 'uplink_speed':100,'weight':2},
                 ]},
              'LAN_4':{'type':'25gbps','topology': 'uplinks_spine_leaf',
        'switch':[{'model':'S_xx64','ports':64,'speed':25, 'height':2,'wattage':300,'uplink_count':4, 'uplink_speed':100,'weight':2},
                {'model':'S_xx32','ports':32,'speed':25, 'height':1,'wattage':200,'uplink_count':2, 'uplink_speed':100,'weight':2},
                 ]},
              'LAN_5':{'type':'1gbps','topology': 'uplinks_spine_leaf',
    'switch':[{'model':'SN_24','ports':24,'speed':1, 'height':1,'wattage':200, 'uplink_count':2, 'uplink_speed':25, 'weight':2},
             ]},
              'LAN_6':{'type':'400GbpsEth','topology':'rails',
    'switch':[{'model':'Z_9xxx','ports':64,'speed':400, 'height':2,'wattage':2200, 'uplink_count':4, 'uplink_speed':800,'weight':20},
             ]},
           }
Cables = {'IB+400':[{'model':'1.5m_400_copper','server_port_speed':400,'split':1,'length':1.5},
                                           {'model':'3m_400_copper','server_port_speed':400,'split':1,'length':3},
                                         {'model':'5m_400_fiber','server_port_speed':400,'split':1,'length':5},
                                      { 'model':'7m_400_fiber','server_port_speed':400,'split':1,'length':7},
                                      { 'model':'10m_400_fiber','server_port_speed':400,'split':1,'length':10},
                                      { 'model':'20m_400_fiber','server_port_speed':400,'split':1,'length':20},
                                      { 'model':'3m_400s_copper','server_port_speed':200,'split':2,'length':3},
                                     {  'model':'5m_400s_fiber','server_port_speed':200,'split':2,'length':5},
                                      { 'model':'7m_400s_fiber','server_port_speed':200,'split':2,'length':7},
                                      { 'model':'10m_400s_fiber','server_port_speed':200,'split':2,'length':10},
                                     {  'model':'20m_400s_fiber','server_port_speed':200,'split':2,'length':20},
                            ],
                            
          'IB+800':[{'model':'1.5m_800_copper','server_port_speed':800,'split':1,'length':1.5},
                                       {'model':'3m_800_copper','server_port_speed':800,'split':1,'length':3},
                                      { 'model':'8m_800_fiber','server_port_speed':800,'split':1,'length':5},
                                      { 'model':'7m_800_fiber','server_port_speed':800,'split':1,'length':7},
                                      { 'model':'10m_800_fiber','server_port_speed':800,'split':1,'length':10},
                                       {'model':'20m_800_fiber','server_port_speed':800,'split':1,'length':20},
                                      { 'model':'3m_800s_copper','server_port_speed':400,'split':2,'length':3},
                                     {  'model':'5m_800s_fiber','server_port_speed':400,'split':2,'length':5},
                                      { 'model':'7m_800s_fiber','server_port_speed':400,'split':2,'length':7},
                                      { 'model':'10m_800s_fiber','server_port_speed':400,'split':2,'length':10},
                                      { 'model':'20m_800s_fiber','server_port_speed':400,'split':2,'length':20},
                   ],
          'S_xx64':[{'model':'1.5m_25_copper','server_port_speed':25,'split':1,'length':1.5},
                                       {'model':'3m_25_copper','server_port_speed':25,'split':1,'length':3},
                                       {'model':'5m_25_fiber','server_port_speed':25,'split':1,'length':5},
                                     {  'model':'7m_25_fiber','server_port_speed':25,'split':1,'length':7},
                                       {'model':'10m_25_fiber','server_port_speed':25,'split':1,'length':10},
                                       {'model':'20m_25_fiber','server_port_speed':25,'split':1,'length':20},                                       
                   ],
          'S_xx32':[{'model':'1.5m_25_copper','server_port_speed':25,'split':1,'length':1.5},
                                       {'model':'3m_25_copper','server_port_speed':25,'split':1,'length':3},
                                       {'model':'5m_25_fiber','server_port_speed':25,'split':1,'length':5},
                                       {'model':'7m_25_fiber','server_port_speed':25,'split':1,'length':7},
                                       {'model':'10m_25_fiber','server_port_speed':25,'split':1,'length':10},
                                       {'model':'20m_25_fiber','server_port_speed':25,'split':1,'length':20},                                       
                   ],
          'SN_24':[{'model':'1.5m_1_copper','server_port_speed':1,'split':1,'length':1.5},
                                      { 'model':'3m_1_coppe','server_port_speed':1,'split':1,'length':3},
                                       {'model':'5m_1_copper','server_port_speed':1,'split':1,'length':5},
                                      { 'model':'7m_1_copper','server_port_speed':1,'split':1,'length':7},
                                      { 'model':'10m_1_copper','server_port_speed':1,'split':1,'length':10},
                                      { 'model':'20m_1_copper','server_port_speed':1,'split':1,'length':20}, 
                                      { 'model':'30m_1_copper','server_port_speed':1,'split':1,'length':30},
                                       {'model':'40m_1_copper','server_port_speed':1,'split':1,'length':40},
                  ],
          'Z_9xxx':[{'model':'1.5m_400_copper_eth','server_port_speed':400,'split':1,'length':1.5},
                                       {'model':'3m_400_copper_eth','server_port_speed':400,'split':1,'length':3},
                                      { 'model':'5m_400_fiber_eth','server_port_speed':400,'split':1,'length':5},
                                      { 'model':'7m_400_fiber_eth','server_port_speed':400,'split':1,'length':7},
                                     {  'model':'10m_400_fiber_eth','server_port_speed':400,'split':1,'length':10},
                                      { 'model':'20m_400_fiber_eth','server_port_speed':400,'split':1,'length':20},
                                      { 'model':'3m_400s_copper_eth','server_port_speed':200,'split':2,'length':3},
                                     {  'model':'5m_400s_fiber_eth','server_port_speed':200,'split':2,'length':5},
                                     {  'model':'7m_400s_fiber_eth','server_port_speed':200,'split':2,'length':7},
                                      { 'model':'10m_400s_fiber_eth','server_port_speed':200,'split':2,'length':10},
                                      { 'model':'20m_400s_fiber_eth','server_port_speed':200,'split':2,'length':20},
                   ],
         }         
            
          

results = []

for rack_config in all_racks_config:
    if rack_config:
        #print(f"Processing rack with config: {rack_config}")
        # Bottom-up placement
        stable_placement_bottom, devices, lans = find_stable_positions_greedy_complex(
            rack_height_u, rack_weight_kg, rack_width_mm, rack_depth_mm,
            rack_config, LANs, Cables, colors_info, prioritize_top=True
        )
        print(f" Top-biased Placement: {stable_placement_bottom}")

        # Top-biased placement
        stable_placement_top_biased, devices, lans = find_stable_positions_greedy_complex(
            rack_height_u, rack_weight_kg, rack_width_mm, rack_depth_mm,
            rack_config, LANs, Cables, colors_info, prioritize_top=False
        )
        print(f" Bottom-up Placement: {stable_placement_top_biased}")
        print("-" * 30)

print("Processing complete.")

In [197]:
def get_rack_layouts(distributions,Cables,LANs, colors_info):
    rack_height_u = 42
    rack_weight_kg = 114.55
    rack_width_mm = 600
    rack_depth_mm = 1200
    distributions_info = dict()
    counter = 1
    for rack_config in distributions:
        if rack_config:
            #print(f"Processing rack with config: {rack_config}")
           
            # Top-biased placement
            
            stable_placement , devices, lans = find_stable_positions_greedy_complex(
                rack_height_u, rack_weight_kg, rack_width_mm, rack_depth_mm,
                rack_config, LANs, Cables, colors_info, prioritize_top=True
            )
            distributions_info[counter] = dict({'rack_config':rack_config, 'stable_placement': stable_placement ,'lan_info':lans,
                                               'device_info':devices})
            counter += 1
            #print(f" Top-biased Placement: {stable_placement_bottom}")

            
             # Bottom-up placement
            #stable_placement_top_biased = find_stable_positions_greedy_complex(
            #    rack_height_u, rack_weight_kg, rack_width_mm, rack_depth_mm,
            #    rack_config,Cables, LANs, colors_info, prioritize_top=false
            #)
            #print(f"   Bottom-up Placement:{stable_placement_top_biased}")
            #print("-" * 30)

    #print("Processing rack layouts Complete.")
    return  distributions_info

In [218]:
import numpy as np
import os
import time
import multiprocessing as mp
from collections import Counter
from itertools import product, permutations
from math import ceil
import logging
import random

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def calculate_box_wattage(box, colors_wattage):
    return sum(box.get(color, 0) * colors_wattage.get(color.split('__')[-1], 0) for color in box)

def calculate_box_height(box, colors_height):
    return sum(box.get(color, 0) * colors_height.get(color.split('__')[-1], 0) for color in box)

def check_box_limits(box, colors_wattage, colors_height, max_box_wattage, max_box_height):
    box_wattage = calculate_box_wattage(box, colors_wattage)
    box_height = calculate_box_height(box, colors_height)
    return box_wattage <= max_box_wattage and box_height <= max_box_height
def get_total_cable_lengths(distributions_info):
    global_minimum_length = 0
    global_actual_length = 0
    cables = dict()
    for rack_info in distributions_info:
        for lan in distributions_info[rack_info]['lan_info']:
            global_minimum_length += distributions_info[rack_info]['lan_info'][lan]['minimum_cable_total_length']
            global_actual_length += distributions_info[rack_info]['lan_info'][lan]['actual_cable_total_length']
            for cable in distributions_info[rack_info]['lan_info'][lan]['cables']:
                if cable not in cables:
                    cables[cable] = 0
                cables[cable] += distributions_info[rack_info]['lan_info'][lan]['cables'][cable]
    
    distributions_info['cables'] = {'global_minimum_length':global_minimum_length, 'global_actual_length':global_actual_length,
                                   'cables':cables}
    
    return distributions_info

def display_distribution_filled_only(distributions_info, colors_wattage, colors_height, total_balls, balls_placed_overall):
    print(f"\nValid Distribution Found (Iterative Greedy - Filled Racks Only):")
    current_total_balls = sum(balls_placed_overall.values())
    total_wattage = 0
    total_height = 0
    total_count = 0
    filled_boxes = []
    for i, info in enumerate(distributions_info):
        if info:
            if 'rack_config' in distributions_info[info]:

                rack_config = distributions_info[info]['rack_config']
                if len(rack_config) == 1 and 'LAN' in str(rack_config):
                    continue
                box_contents = ", ".join(f"{count} {color}" for color, count in rack_config.items())

                box_wattage = calculate_box_wattage(rack_config, colors_wattage)

                box_height = calculate_box_height(rack_config, colors_height)

                total_wattage += box_wattage
                total_height += box_height
                ball_count = sum(rack_config.values())
                total_count += ball_count
                filled_boxes.append(f"\x1b[1m -------------------------Rack {i+1} info------------------------------\x1b[0m")
                filled_boxes.append(f"\x1b[1;4m-->Rack {i+1}:\x1b[0m \n {box_contents} (Wattage: {box_wattage}, Height: {box_height}, count:{ball_count})")
                filled_boxes.append(f"-->\x1b[1;4mdevice placement: in the rack \x1b[0m \n {distributions_info[info]['stable_placement']}")
                filled_boxes.append(f"-->\x1b[1;4maggregated LAN info: \x1b[0m \n {distributions_info[info]['lan_info']}")
                filled_boxes.append(f"-->\x1b[1;4mDevice detailed info: \x1b[0m \n {distributions_info[info]['device_info']}")
            else:
                filled_boxes.append(f"\x1b[1;4m-------------------------Total distribution info------------------------------ \x1b[0m \n {distributions_info[info]}")
    if filled_boxes:
        for box_info in filled_boxes:
            print(box_info)
        print(f"  Total Wattage: {total_wattage}, Total Height: {total_height}, Total Balls Distributed:{total_count} {current_total_balls}")
    else:
        print(f"  No racks were filled. Total Balls Distributed: {current_total_balls}")
    if current_total_balls != total_balls:
        print(f"  WARNING: Total balls distributed ({current_total_balls}) does not match the expected total ({total_balls})!")
    

def is_switch_relevent(rack, switch_info, colors_info):
    alllans = set()
    allnodes = ""
    include_lan = set()
    execlude_lan = set()
    for device in rack:
        if 'LAN' in device:
            switch = device.split('__')[-1]
            lan = device.split('__')[0]
            alllans.add(lan)
    for device in rack:
        if 'LAN' not in device:
            for lan in alllans:
                if colors_info[device][lan]['count'] > 0:
                    include_lan.add(lan)
    execlude_lan = alllans - include_lan
    
    return list(execlude_lan)

def get_distribution_signature(distribution):
        filled_boxes_signature = tuple(sorted(tuple(sorted(box.items())) for box in distribution if box))
        return filled_boxes_signature

def find_valid_distributions_iterative_greedy_adaptive(LANs, Cables, switch_info, colors_info, initial_num_boxes, max_box_wattage, max_box_height, total_balls):
    colors = list(colors_info.keys())
    colors_wattage = {color: colors_info[color]['wattage'] for color in colors}
    colors_wattage.update({switch_info[lan][0]['model']: switch_info[lan][0]['wattage'] for lan in list(switch_info.keys())})
    
    colors_height = {color: colors_info[color]['height'] for color in colors}
    colors_height.update({switch_info[lan][0]['model']: switch_info[lan][0]['height'] for lan in list(switch_info.keys())})
    
    max_possible_boxes = total_balls
    valid_distributions = []
    seen_distributions = set()

    

    num_boxes_to_try = list(range(initial_num_boxes, max_possible_boxes + 1))
    random.shuffle(num_boxes_to_try)
    
    
    switch_scenarios = []
    for lan in switch_info:
        switch_scenarios.append(("switch_per_rack", lan))
        switch_scenarios.append(("switch_per_twoRacks_1st", lan))
        switch_scenarios.append(("switch_per_twoRacks_2nd", lan))
        switch_scenarios.append(("switch_per_3Racks_1st", lan))
        switch_scenarios.append(("switch_per_3Racks_2nd", lan))
        switch_scenarios.append(("switch_per_3Racks_3rd", lan))
    swtich_index = 0
    current_switches = switch_info.copy()
    if switch_scenarios[0][0] == 'switch_per_rack':
        switch_to_add = switch_scenarios[0][1] +'__'+ switch_info[switch_scenarios[0][1]][0]['model']
    swcounter = 0   
    for sw_scenario,lan_type in switch_scenarios:
        for num_boxes in num_boxes_to_try:
            distributions = [Counter() for _ in range(num_boxes)]
            switch_to_add = lan_type + '__' + switch_info[lan_type][0]['model']
            if sw_scenario == 'switch_per_rack':
                for box in distributions:
                    box[switch_to_add] = 1
            balls_placed = {color: 0 for color in colors}
            box_index = 0
            while box_index < num_boxes and any(balls_placed[color] < colors_info[color]['count'] for color in colors):
                current_box = distributions[box_index]

                remaining_balls = {c: colors_info[c]['count'] - balls_placed[c] for c in colors}
                possible_colors = [c for c in colors if remaining_balls[c] > 0]

                if not possible_colors:
                    box_index += 1
                    continue

                scenarios = []

                for color in possible_colors:
                    scenarios.append(("single_color", color))
                    scenarios.append(("partial_color_50", color))
                    scenarios.append(("partial_color_25", color))
                    scenarios.append(("partial_color_10", color))
                    scenarios.append(("mixed_fill_initial", color))

                random.shuffle(scenarios) # Try scenarios in a random order for each box

                applied_scenario = False
                for scenario_type, main_color in scenarios:
                    temp_box = current_box.copy()
                    temp_balls_placed = balls_placed.copy()
                    temp_remaining_balls = remaining_balls.copy()

                    if scenario_type == "single_color":
                        while temp_remaining_balls.get(main_color, 0) > 0 and check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                                calculate_box_wattage(temp_box, colors_wattage) + colors_wattage.get(main_color, 0) <= max_box_wattage and \
                                calculate_box_height(temp_box, colors_height) + colors_height.get(main_color, 0) <= max_box_height:
                            temp_box[main_color] = temp_box.get(main_color, 0) + 1
                            temp_balls_placed[main_color] = temp_balls_placed.get(main_color, 0) + 1
                            temp_remaining_balls[main_color] -= 1

                        for other_color in list(temp_remaining_balls.keys()):
                            if other_color != main_color and temp_remaining_balls.get(other_color, 0) > 0:
                                while temp_remaining_balls[other_color] > 0 and check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                                        calculate_box_wattage(temp_box, colors_wattage) + colors_wattage.get(other_color, 0) <= max_box_wattage and \
                                        calculate_box_height(temp_box, colors_height) + colors_height.get(other_color, 0) <= max_box_height:
                                    temp_box[other_color] = temp_box.get(other_color, 0) + 1
                                    temp_balls_placed[other_color] = temp_balls_placed.get(other_color, 0) + 1
                                    temp_remaining_balls[other_color] -= 1


                        if temp_box != current_box:
                            distributions[box_index] = temp_box
                            balls_placed.update(temp_balls_placed)
                            applied_scenario = True
                            break

                    elif scenario_type.startswith("partial_color"):
                        percentage = float(int(scenario_type.split('_')[2])/100)
                        add_amount = min(ceil(colors_info[main_color]['count'] * percentage), temp_remaining_balls[main_color])
                        can_add = True
                        for _ in range(add_amount):
                            if check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                               calculate_box_wattage(temp_box, colors_wattage) + colors_wattage[main_color] <= max_box_wattage and \
                               calculate_box_height(temp_box, colors_height) + colors_height[main_color] <= max_box_height and \
                               temp_remaining_balls[main_color] > 0:
                                temp_box[main_color] += 1
                                temp_balls_placed[main_color] += 1
                                temp_remaining_balls[main_color] -= 1
                            else:
                                can_add = False
                                break
                        if any(count > 0 for color, count in temp_remaining_balls.items() if color != main_color):
                            for other_color in list(temp_remaining_balls.keys()):
                                if other_color != main_color and temp_remaining_balls.get(other_color, 0) > 0:

                                    while temp_remaining_balls[other_color] > 0 and check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                                            calculate_box_wattage(temp_box, colors_wattage) + colors_wattage.get(other_color, 0) <= max_box_wattage and \
                                            calculate_box_height(temp_box, colors_height) + colors_height.get(other_color, 0) <= max_box_height:

                                        temp_box[other_color] = temp_box.get(other_color, 0) + 1
                                        temp_balls_placed[other_color] = temp_balls_placed.get(other_color, 0) + 1
                                        temp_remaining_balls[other_color] -= 1


                        if (can_add or add_amount > 0) and temp_box != current_box:
                            distributions[box_index] = temp_box
                            balls_placed.update(temp_balls_placed)
                            applied_scenario = True
                            break

                    elif scenario_type == "mixed_fill_initial":
                        while temp_remaining_balls[main_color] > 0 and check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                                calculate_box_wattage(temp_box, colors_wattage) + colors_wattage[main_color] <= max_box_wattage and \
                                calculate_box_height(temp_box, colors_height) + colors_height[main_color] <= max_box_height:
                            temp_box[main_color] += 1
                            temp_balls_placed[main_color] += 1
                            temp_remaining_balls[main_color] -= 1

                        other_colors = [c for c in possible_colors if c != main_color]
                        random.shuffle(other_colors)
                        while check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                                calculate_box_wattage(temp_box, colors_wattage) < max_box_wattage and \
                                calculate_box_height(temp_box, colors_height) < max_box_height and \
                                any(temp_remaining_balls[c] > 0 for c in other_colors):
                            added_other = False
                            for other_color in other_colors:
                                if temp_remaining_balls[other_color] > 0 and \
                                   check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                                   calculate_box_wattage(temp_box, colors_wattage) + colors_wattage[other_color] <= max_box_wattage and \
                                   calculate_box_height(temp_box, colors_height) + colors_height[other_color] <= max_box_height:
                                    temp_box[other_color] += 1
                                    temp_balls_placed[other_color] += 1
                                    temp_remaining_balls[other_color] -= 1
                                    added_other = True
                                    break
                            if not added_other:
                                break
                        if temp_box != current_box:
                            distributions[box_index] = temp_box
                            balls_placed.update(temp_balls_placed)
                            applied_scenario = True
                            break
                advance_box = 0        
                if applied_scenario or not possible_colors:
                    advance_box = 1
                elif not any(remaining_balls.values()):
                    advance_box = 1
                if advance_box:

                    execlude_switch = is_switch_relevent(temp_box,switch_info,colors_info)
                    if execlude_switch:
                        advanece_box = 0
                        for lan in execlude_switch:
                            temp_box = Counter({k: v for k, v in temp_box.items() if lan not in k})
                            distributions[box_index] = temp_box

                    box_index += 1

            if sum(balls_placed.values()) == total_balls:
                signature = get_distribution_signature(distributions)
                if signature not in seen_distributions:
                    seen_distributions.add(signature)
                    distributions_info = get_rack_layouts(distributions,Cables, LANs, colors_info)
                    
                    distributions_info = get_total_cable_lengths(distributions_info)
                    
                    display_distribution_filled_only(distributions_info, colors_wattage, colors_height, total_balls, balls_placed)
                    valid_distributions.append(list(distributions))
                    # Potentially break here if you only need one solution
                    return valid_distributions

    return valid_distributions

In [221]:
def main_iterative_greedy_adaptive_with_height():
    colors_info = {
        'compute_nodes': {'count': 20, 'wattage': 1600, 'height': 2, 'weight':30, 'LAN_1':{'count':1,'speed':400},
                                                                           'LAN_2':{'count':1,'speed':200},
                                                                           'LAN_3':{'count':1,'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                            'LAN_6':{'count':0, 'speed':400},
                      },
        'GPU_nodes': {'count': 20, 'wattage': 11000, 'height': 8, 'weight': 15, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':1,'speed':200},
                                                                           'LAN_3':{'count':1, 'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                            'LAN_6':{'count':8, 'speed':400},
                      },
        'storage_nodes': {'count': 30, 'wattage': 1250, 'height': 2, 'weight':20, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':3,'speed':200},
                                                                           'LAN_3':{'count':1, 'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                          'LAN_6':{'count':0, 'speed':400},
                         },
        'FrontEnd_nodes': {'count': 30, 'wattage': 750, 'height': 1, 'weight':9, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':0,'speed':200},
                                                                            'LAN_3':{'count':1, 'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                         'LAN_6':{'count':0, 'speed':400},
                          },
                   
    }
    LANs = { 'LAN_1':{'type':'400gbsNDR','topology': 'halfports_spine_leaf',
                  'switch':[{'model':'IB+400','ports':64,'speed':400, 'height':2,'wattage':2000, 'weight':20},
                         {'model':'IB_800','ports':64,'speed':800, 'height':2,'wattage':2000, 'weight':20},
                           ]},
              'LAN_2':{'type':'400GbpsEth', 'topology': 'uplinks_spine_leaf',
'switch':[{'model':'Z_9xxx','ports':64,'speed':400, 'height':2,'wattage':2200, 'uplink_count':4, 'uplink_speed':800,'weight':20},
                 {'model':'Z_6xxx','ports':64,'speed':200, 'height':1,'wattage':2000, 'uplink_count':4, 'uplink_speed':400,'weight':20},
        ]},
              'LAN_3':{'type':'25gbps','topology': 'uplinks_spine_leaf',
        'switch':[{'model':'S_xx64','ports':64,'speed':25, 'height':2,'wattage':300, 'uplink_count':4, 'uplink_speed':100,'weight':2},
                {'model':'S_xx32','ports':32,'speed':25, 'height':1,'wattage':200,'uplink_count':4, 'uplink_speed':100,'weight':2},
                 ]},
              'LAN_4':{'type':'25gbps','topology': 'uplinks_spine_leaf',
        'switch':[{'model':'S_xx64','ports':64,'speed':25, 'height':2,'wattage':300,'uplink_count':4, 'uplink_speed':100,'weight':2},
                {'model':'S_xx32','ports':32,'speed':25, 'height':1,'wattage':200,'uplink_count':2, 'uplink_speed':100,'weight':2},
                 ]},
              'LAN_5':{'type':'1gbps','topology': 'uplinks_spine_leaf',
    'switch':[{'model':'SN_24','ports':24,'speed':1, 'height':1,'wattage':200, 'uplink_count':2, 'uplink_speed':25, 'weight':2},
             ]},
              'LAN_6':{'type':'400GbpsEth','topology':'rails',
    'switch':[{'model':'Z_9xxx','ports':64,'speed':400, 'height':2,'wattage':2200, 'uplink_count':4, 'uplink_speed':800,'weight':20},
             ]},
           }
    Cables = {'IB+400':[{'model':'1.5m_400_copper','server_port_speed':400,'split':1,'length':1.5},
                                           {'model':'3m_400_copper','server_port_speed':400,'split':1,'length':3},
                                         {'model':'5m_400_fiber','server_port_speed':400,'split':1,'length':5},
                                      { 'model':'7m_400_fiber','server_port_speed':400,'split':1,'length':7},
                                      { 'model':'10m_400_fiber','server_port_speed':400,'split':1,'length':10},
                                      { 'model':'20m_400_fiber','server_port_speed':400,'split':1,'length':20},
                                      { 'model':'3m_400s_copper','server_port_speed':200,'split':2,'length':3},
                                     {  'model':'5m_400s_fiber','server_port_speed':200,'split':2,'length':5},
                                      { 'model':'7m_400s_fiber','server_port_speed':200,'split':2,'length':7},
                                      { 'model':'10m_400s_fiber','server_port_speed':200,'split':2,'length':10},
                                     {  'model':'20m_400s_fiber','server_port_speed':200,'split':2,'length':20},
                            ],
                            
          'IB+800':[{'model':'1.5m_800_copper','server_port_speed':800,'split':1,'length':1.5},
                                       {'model':'3m_800_copper','server_port_speed':800,'split':1,'length':3},
                                      { 'model':'8m_800_fiber','server_port_speed':800,'split':1,'length':5},
                                      { 'model':'7m_800_fiber','server_port_speed':800,'split':1,'length':7},
                                      { 'model':'10m_800_fiber','server_port_speed':800,'split':1,'length':10},
                                       {'model':'20m_800_fiber','server_port_speed':800,'split':1,'length':20},
                                      { 'model':'3m_800s_copper','server_port_speed':400,'split':2,'length':3},
                                     {  'model':'5m_800s_fiber','server_port_speed':400,'split':2,'length':5},
                                      { 'model':'7m_800s_fiber','server_port_speed':400,'split':2,'length':7},
                                      { 'model':'10m_800s_fiber','server_port_speed':400,'split':2,'length':10},
                                      { 'model':'20m_800s_fiber','server_port_speed':400,'split':2,'length':20},
                   ],
          'S_xx64':[{'model':'1.5m_25_copper','server_port_speed':25,'split':1,'length':1.5},
                                       {'model':'3m_25_copper','server_port_speed':25,'split':1,'length':3},
                                       {'model':'5m_25_fiber','server_port_speed':25,'split':1,'length':5},
                                     {  'model':'7m_25_fiber','server_port_speed':25,'split':1,'length':7},
                                       {'model':'10m_25_fiber','server_port_speed':25,'split':1,'length':10},
                                       {'model':'20m_25_fiber','server_port_speed':25,'split':1,'length':20},                                       
                   ],
          'S_xx32':[{'model':'1.5m_25_copper','server_port_speed':25,'split':1,'length':1.5},
                                       {'model':'3m_25_copper','server_port_speed':25,'split':1,'length':3},
                                       {'model':'5m_25_fiber','server_port_speed':25,'split':1,'length':5},
                                       {'model':'7m_25_fiber','server_port_speed':25,'split':1,'length':7},
                                       {'model':'10m_25_fiber','server_port_speed':25,'split':1,'length':10},
                                       {'model':'20m_25_fiber','server_port_speed':25,'split':1,'length':20},                                       
                   ],
          'SN_24':[{'model':'1.5m_1_copper','server_port_speed':1,'split':1,'length':1.5},
                                      { 'model':'3m_1_coppe','server_port_speed':1,'split':1,'length':3},
                                       {'model':'5m_1_copper','server_port_speed':1,'split':1,'length':5},
                                      { 'model':'7m_1_copper','server_port_speed':1,'split':1,'length':7},
                                      { 'model':'10m_1_copper','server_port_speed':1,'split':1,'length':10},
                                      { 'model':'20m_1_copper','server_port_speed':1,'split':1,'length':20}, 
                                      { 'model':'30m_1_copper','server_port_speed':1,'split':1,'length':30},
                                       {'model':'40m_1_copper','server_port_speed':1,'split':1,'length':40},
                  ],
          'Z_9xxx':[{'model':'1.5m_400_copper_eth','server_port_speed':400,'split':1,'length':1.5},
                                       {'model':'3m_400_copper_eth','server_port_speed':400,'split':1,'length':3},
                                      { 'model':'5m_400_fiber_eth','server_port_speed':400,'split':1,'length':5},
                                      { 'model':'7m_400_fiber_eth','server_port_speed':400,'split':1,'length':7},
                                     {  'model':'10m_400_fiber_eth','server_port_speed':400,'split':1,'length':10},
                                      { 'model':'20m_400_fiber_eth','server_port_speed':400,'split':1,'length':20},
                                      { 'model':'3m_400s_copper_eth','server_port_speed':200,'split':2,'length':3},
                                     {  'model':'5m_400s_fiber_eth','server_port_speed':200,'split':2,'length':5},
                                     {  'model':'7m_400s_fiber_eth','server_port_speed':200,'split':2,'length':7},
                                      { 'model':'10m_400s_fiber_eth','server_port_speed':200,'split':2,'length':10},
                                      { 'model':'20m_400s_fiber_eth','server_port_speed':200,'split':2,'length':20},
                   ],
         }         
            
          
    
    num_devices = sum(info['count'] for info in colors_info.values())
    max_needed_wattage = sum(colors_info[color]['wattage'] * colors_info[color]['count'] for color in colors_info)
    max_box_wattage = 20000
    max_box_height = 42
    # arrange the list of lANs that are found int eh color_info
    current_lans = []
    str_nodes= str(colors_info)
    for lan in LANs:
        if lan in str_nodes:
            current_lans.append(lan)
   
    #create a dictionary same like the color_info which include the LANs various switches
    switch_info = dict()
    for lan in current_lans:
        switch_info[lan] = LANs[lan]['switch'].copy()
            
    #share this dictionary with the find_valid_distributions function
    initial_num_boxes = ceil(max_needed_wattage / max_box_wattage) + 2

    print(f"Distributing {num_devices} balls into initially {initial_num_boxes} boxes which consumes {max_needed_wattage} watts (Iterative Greedy Adaptive with Height):")

    start_time = time.time()
    distributions = find_valid_distributions_iterative_greedy_adaptive(
        LANs,Cables, switch_info,  colors_info, initial_num_boxes, max_box_wattage, max_box_height, num_devices
    )
    end_time = time.time()

    print(f"\nSummary: Found {len(distributions)} unique distribution(s) in {end_time - start_time:.6f} seconds")

if __name__ == "__main__":
    main_iterative_greedy_adaptive_with_height()

Distributing 100 balls into initially 18 boxes which consumes 312000 watts (Iterative Greedy Adaptive with Height):

Valid Distribution Found (Iterative Greedy - Filled Racks Only):
 -------------------------Rack 1 info------------------------------
-->Rack 1: 
 1 LAN_1__IB+400, 1 GPU_nodes, 4 compute_nodes (Wattage: 19400, Height: 18, count:6)
-->device placement: in the rack  
 {'LAN_1__IB+400_1': 41, 'GPU_nodes_2': 33, 'compute_nodes_3': 31, 'compute_nodes_4': 29, 'compute_nodes_5': 27, 'compute_nodes_6': 25}
-->aggregated LAN info:  
 {'LAN_1__IB+400_1': {'model': 'IB+400', 'ports': 64, 'speed': 400, 'height': 2, 'wattage': 2000, 'weight': 20, 'type': 'LAN_1__IB+400', 'position': 41, 'id': '1', 'half_for_split': 0, 'full_no_split': 4, 'cables': {'1.5m_400_copper': 4}, 'minimum_cable_total_length': 4.7170000000000005, 'actual_cable_total_length': 6.0}}
-->Device detailed info:  
 {'compute_nodes_3': {'position': 31, 'id': '3', 'LAN_1__IB+400_1': {'cable_count': 1, 'speed': 400, 'ful

In [ ]:
x = 'zeko'.split('_')[-1]

In [ ]:
print(x.split('_')[-1])